# 2492 — EIC API Technical Discovery
## AI-Enhanced Integration of External O&G Market Databases into Salesforce

## EIC Members' API — Official Documentation (v1, October 2025)

| Property | Value |
| --- | --- |
| **Base URL** | `https://memberapi.the-eic.com` |
| **API Version** | v1 (also v2 available) |
| **Discovery** | `GET https://memberapi.the-eic.com/v1` (no auth needed) |
| **Auth Header** | `Authorization: key <apiKey>` |
| **Style** | REST, GET with querystring or POST with JSON body |
| **Response** | JSON |
| **Key Manager** | Self-service via EIC DataStream Tools menu |
| **Annual Fee** | £3,500 + VAT |

### Authentication

The API key is stored in Azure Key Vault and exposed to Databricks through secret scope `WE-SPOKE-MDP-AUT-T-KV` as `EIC-API`. Load it with `dbutils.secrets.get`; do not store or print the key in notebook cells or output. Key rotation is managed in EIC DataStream's API Key Manager.

### Discovery (no auth required)

```text
GET https://memberapi.the-eic.com
GET https://memberapi.the-eic.com/v1
```

These endpoints describe the supported API versions and parameters.

## Official EIC Members' API Endpoints

### Core Endpoints
| Method | Endpoint | Auth | Purpose |
| --- | --- | --- | --- |
| GET | `/v1` | ❌ No | Discovery — describes all endpoints & parameters |
| GET | `/v1/projects/{projectId}` | ✅ Yes | **Single project** (+ data shaping params) |
| GET | `/v1/projects?<selection>&<projection>` | ✅ Yes | **Multiple projects** by criteria |
| POST | `/v1/projects/search` | ✅ Yes | Same as above, JSON body |
| GET | `/v1/projectnames?<selection>` | ✅ Yes | **Project name list** (lightweight) |
| POST | `/v1/projectnames/search` | ✅ Yes | Same as above, JSON body |

### Lookup Endpoints (for IDs)
| Method | Endpoint | Returns |
| --- | --- | --- |
| GET | `/v1/energysectors` | All energy sectors with IDs |
| GET | `/v1/energysectors/{id}/projectTypes` | Project types for a sector |
| GET | `/v1/projecttypes` | All project types |
| GET | `/v1/projectstages` | All project stages |
| GET | `/v1/projectstatuses` | All project statuses |

### Selection Parameters (filter which projects)
| Parameter | Type | Required | Description |
| --- | --- | --- | --- |
| `energySector` | int | ✅ Required* | 1=Upstream, etc. |
| `projectTypes` | int[] | Optional | Filter by type |
| `projectStages` | int[] | Optional | Filter by stage |
| `projectStatuses` | int[] | Optional | Filter by status |
| `updatedSince` | date | Optional | Only recently updated |
| `isOnshore` | bool | Optional | Terrain filter |
| `isOffshore` | bool | Optional | Terrain filter |

\* Required for multi-project queries in v1; v2 may differ

### Data Shaping Parameters (enrich response)
| Parameter | Type | Description |
| --- | --- | --- |
| `splitUk` | bool | Split UK into nations |
| `includeFid` | bool | FID expected/received dates |
| `includeAllLocations` | bool | All locations array (not just main) |
| `includeOperators` | bool | All operating companies + equity share |
| `includeContracts` | bool | **Contracts + contractors + bidders** 🎯 |
| `includeProjectNotes` | int (max 30) | Recent project notes |
| `includeProjectNotesSince` | date | Notes since date |
| `includeEquipment` | bool | All equipment for sector |
| `includePlatforms` | bool | Platform data |
| `includePipelines` | bool | Pipeline data |
| `includeHydrogen` | bool | Hydrogen capacity |
| `includeFields` | bool | Upstream field data |
| `includeWindTurbines` | bool | Wind turbine data |
| `includeSolar` | bool | Solar farm data |
| `includeElectricalCables` | bool | Cable routing |
| `includeFloatingUnits` | bool | FPSO/FSO data |
| `includePlant` | bool | Plant infrastructure |
| `includeStorage` | bool | Storage facilities |
| `includeTerminals` | bool | Terminal infrastructure |
| `includeSurf` | bool | Subsea umbilicals/risers/flowlines |
| `includeLngPlants` | bool | LNG plant data |
| `includeWells` | bool | Well data |

### ProjectNames Response Model
```json
{
  "projectId": 12345,
  "name": "Brazil South Atlantic FPSO Phase 2",
  "created": "2024-03-15T00:00:00Z",
  "updated": "2026-02-01T00:00:00Z",
  "_links": {
    "eicDataStream": {"url": "https://eicdatastream.the-eic.com/...", "method": "GET"},
    "project": {"url": "https://memberapi.the-eic.com/v1/projects/12345", "method": "GET"}
  }
}
```

### Key Insight for Santiago's Use Case
✅ `includeContracts=true` → Returns **contractors + bidders** per project (exactly what Santiago needs for EPC identification!)  
✅ `includeOperators=true` → Returns **all operating companies + equity share**  
✅ `updatedSince` → Enables **incremental ingestion** (only new/changed projects)  
✅ `includeFid=true` → FID dates for timing intelligence

## Project Data Model (Response Fields)

*Extracted from codebase analysis — field name: occurrence count in JS bundle*

### Core Project Fields
| Field | Occurrences | Description | Santiago's Use |
| --- | ---: | --- | --- |
| `projectID` | 215 | Unique project identifier | Primary key |
| `projectName` | 188 | Project name | Display & matching |
| `country` / `countryName` | 130 / 18 | Project country | Region assignment |
| `region` | 74 | Geographic region | Team allocation |
| `stage` / `projectStage` | 62 / 46 | FEED, EPC, Conceptual, etc. | Priority filter |
| `startupYear` | 70 | Expected startup year | Timeline planning |
| `operator` / `operatorName` | 50 | Operating company | Stakeholder ID |
| `sector` / `industrySector` | 40 / 32 | Downstream, Upstream, etc. | Segment filter |
| `contractor` | 34 | EPC/construction company | **KEY: customer target** |
| `capex` / `dollarValue` | 17 / 29 | Capital expenditure (USD millions) | Revenue calculation |
| `capacity` / `capacityUnit` | 34 / 4 | Production capacity | Project sizing |
| `projectType` | 24 | Project classification | Filtering |
| `projectStatus` | 23 | Active/cancelled/etc. | Status tracking |
| `terrain` | 18 | Offshore / Onshore | Technical scope |
| `waterDepth` | 60 | Offshore depth | Technical scope |
| `fidDate` | 32 | Final Investment Decision | Timing indicator |
| `description` | 84 | Project description | AI enrichment source |
| `lastUpdated` | 1 | Last data update | Freshness |

### Project Sub-Resources
| Resource | Fields (inferred) |
| --- | --- |
| Operating Companies | company name, role, country |
| Project Contracts | contractor, contract type, stage, value |
| Project Locations | coordinates, country, locality |
| Project Links | related URLs/sources |
| Related Projects | linked project IDs |

### Santiago's Key Use Case Fields
From the meeting, Santiago specifically mentioned:
1. **CAPEX** → "I have a calculation to know how much potential business for Promat"
2. **Stage** (FEED/EPC/Conceptual) → "red is FEED = engineering stage, EPC = advanced = our customers"
3. **Stakeholders** (EPCs, operators) → "which companies are behind it, construction and engineering ones"
4. **Country/Region** → "I assign per region to each of my teammates"
5. **Startup Year** → "I want to target next year projects, so I go to 2027"
6. **Sector** → Downstream = refineries (primary), also hydrogen, LNG, renewables

## Search Criteria Model (Query Filters)

### Available Search Filters
| Filter | Type | Example |
| --- | --- | --- |
| `selectedIndustrySectors` / `selectedIndustrySectorIDs` | Multi-select | Downstream, Upstream, Renewables |
| `selectedCountries` | Multi-select | Brazil, Algeria, Korea |
| `selectedRegions` | Multi-select | South America, MENA |
| `selectedOperators` | Multi-select | Company names |
| `selectedContractors` | Multi-select | EPC company names |
| `selectedProjectTypes` | Multi-select | Type classification |
| `selectedEquipmentTypes` | Multi-select | Equipment |
| `selectedBiddingStatuses` | Multi-select | Bidding stage |
| `dollarValueFrom` / `dollarValueTo` | Range (USD) | 100M - 5000M |
| `startupYearFrom` / `startupYearTo` | Range (year) | 2026 - 2032 |
| `megawattsFrom` / `megawattsTo` | Range (capacity) |
| `waterDepthFrom` / `waterDepthTo` | Range (meters) |
| `fidDateFrom` / `fidDateTo` | Date range | FID dates |
| `includeProjectsWithFidData` | Boolean | |
| `includeFidReceived` | Boolean | |
| `projectOffshore` / `projectOnshore` | Boolean | Terrain filter |
| `createdWithinWeeks` | Integer | Recently added filter |
| `contractAwardDateFrom` / `contractAwardDateTo` | Date range | |
| `locationMode` | String | Location filter mode |

### EIC v1 API list behavior (verified 28 September 2026)

`POST /v1/projects/search` and `POST /v1/projectnames/search` returned the entire Upstream sector (1,747 IDs) at offsets 0, 100, 1,600, and 1,747 with `paging.limit=100`. The API did not apply the supplied `paging` object or return `totalCount`. Fetch each sector once and compare the returned project IDs against its independently requested project-name index. Do not keep incrementing an ignored offset; it will repeat the same results.

The UI search/sorting model below came from a separate application and must not be assumed to be the Members' API v1 contract.

### Export Request Body (separate UI)
```json
{
  "fileName": "promat_og_projects.xlsx",
  "includeContractors": true,
  "includeBidders": true,
  "epcOnly": false,
  "contracts": "All"
}
```

### Industry Sectors (confirmed from the API)
- `upstream` — Oil/gas extraction (ID 1)
- `midstream` — Transportation/infrastructure (ID 7)
- `downstream` — Refineries, petrochemical (ID 2; Santiago's primary focus)

### Project Stages (from Santiago's demo)
- **Conceptual** — Very early design
- **FEED** — Front-End Engineering Design (engineering stage)
- **EPC** — Engineering, Procurement, Construction (advanced — EPCs are Promat's customers)
- **Construction** — Active building
- **Completed** — Operational

In [0]:
# =============================================================================
# EIC Members' API — Production-Ready Ingestion Template
# =============================================================================
# Base URL: https://memberapi.the-eic.com
# Auth:     Authorization: key <apiKey>
# Docs:     GET https://memberapi.the-eic.com/v1 (no auth needed)
# Status:   API key stored in Azure Key Vault-backed Databricks secret scope
# =============================================================================

import requests
import json
import urllib3
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

# POC: Disable SSL verification (Databricks Serverless corporate proxy issue)
# TODO: Replace with proper CA bundle in production
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# --- Configuration ---
EIC_BASE_URL = "https://memberapi.the-eic.com"
API_KEY = dbutils.secrets.get(scope="WE-SPOKE-MDP-AUT-T-KV", key="EIC-API")

HEADERS = {
    "Authorization": f"key {API_KEY}",
    "Accept": "application/json"
}

print("EIC API key loaded from Key Vault")

# --- Helper Functions ---
def eic_get(endpoint, params=None):
    """GET request to EIC Members' API."""
    url = f"{EIC_BASE_URL}{endpoint}"
    response = requests.get(url, headers=HEADERS, params=params, timeout=60, verify=False)
    response.raise_for_status()
    return response.json()

def eic_post(endpoint, body):
    """POST request to EIC Members' API."""
    url = f"{EIC_BASE_URL}{endpoint}"
    headers = {**HEADERS, "Content-Type": "application/json"}
    response = requests.post(url, headers=headers, json=body, timeout=60, verify=False)
    response.raise_for_status()
    return response.json()

# =============================================================================
# Step 1: Get Lookup Tables (understand available IDs)
# =============================================================================
print("--- Step 1: Loading Lookup Tables ---")
try:
    energy_sectors = eic_get("/v1/energysectors")
    print(f"✅ Energy Sectors: {json.dumps(energy_sectors, indent=2)}")
    
    project_types = eic_get("/v1/projecttypes")
    print(f"\n✅ Project Types ({len(project_types)} types): {json.dumps(project_types[:5], indent=2)}...")
    
    project_stages = eic_get("/v1/projectstages")
    print(f"\n✅ Project Stages: {json.dumps(project_stages, indent=2)}")
    
    project_statuses = eic_get("/v1/projectstatuses")
    print(f"\n✅ Project Statuses: {json.dumps(project_statuses, indent=2)}")
    
except requests.exceptions.HTTPError as e:
    if e.response.status_code == 401:
        print(f"❌ AUTH FAILED: {e.response.json()}")
        print("   → Need valid API key. Ask Santiago for the test key.")
    else:
        print(f"❌ API Error: {e.response.status_code} - {e.response.text}")
except Exception as e:
    print(f"❌ Connection Error: {e}")

# =============================================================================
# Step 2: Discover Sector IDs + Fetch Downstream Projects
# =============================================================================
# Note: lookup endpoint returned energySectorId=0 for all (test API quirk).
# We probe integer IDs 1-7 to find which maps to Downstream.
print("\n--- Step 2: Sector ID discovery ---")
downstream_id = None
for sid in range(1, 8):
    try:
        r = requests.get(
            f"{EIC_BASE_URL}/v1/projects",
            headers={**HEADERS, "Accept": "application/json"},
            params={"energySector": sid, "paging.limit": 1},
            timeout=20, verify=False
        )
        if r.status_code == 200:
            data = r.json()
            projects = data if isinstance(data, list) else data.get("projects", data.get("data", []))
            total = data.get("totalCount", len(projects)) if isinstance(data, dict) else len(data)
            sector_name = ""
            if projects:
                p = projects[0]
                sector_name = p.get("industrySector", p.get("sector", p.get("energySector", "")))
            print(f"  sector_id={sid}: total={total} | sector='{sector_name}'")
            if "downstream" in str(sector_name).lower():
                downstream_id = sid
                print(f"  ✅ Downstream = sector_id {sid}")
        else:
            print(f"  sector_id={sid}: HTTP {r.status_code} — {r.text[:80]}")
    except requests.exceptions.HTTPError as e:
        print(f"  sector_id={sid}: {e.response.status_code} — {e.response.text[:80]}")
    except Exception as e:
        print(f"  sector_id={sid}: {e}")

# =============================================================================
# Step 3: Fetch Single Project — Full Schema Discovery
# =============================================================================
print("\n--- Step 3: Full schema (project 2647) ---")
try:
    project = eic_get("/v1/projects/2647", params={
        "includeOperators": "true",
        "includeContracts": "true",
        "includeFid": "true",
        "includeAllLocations": "true",
        "includeProjectNotes": "5"
    })
    print(f"✅ Project fields: {list(project.keys())}")
    print(json.dumps(project, indent=2)[:3000])
except requests.exceptions.HTTPError as e:
    print(f"❌ {e.response.status_code}: {e.response.text[:200]}")
except Exception as e:
    print(f"❌ {e}")

# =============================================================================
# Step 4: Sample Downstream Projects (Santiago's use case)
# =============================================================================
if downstream_id:
    print(f"\n--- Step 4: Downstream projects (sector_id={downstream_id}) ---")
    try:
        result = eic_post("/v1/projects/search", {
            "energySector": downstream_id,
            "paging": {"limit": 5, "offset": 0},
            "includeOperators": True,
            "includeContracts": True,
            "includeFid": True,
            "includeAllLocations": True
        })
        projects = result if isinstance(result, list) else result.get("projects", result.get("data", [result]))
        print(f"✅ Sample: {len(projects)} downstream projects")
        for p in projects[:3]:
            print(f"  - [{p.get('projectID','?')}] {p.get('projectName','?')} | {p.get('country','?')} | {p.get('stage','?')} | CAPEX: ${p.get('dollarValue', p.get('capex','?'))}M")
        print("\nFull first project:")
        print(json.dumps(projects[0] if projects else {}, indent=2)[:2000])
    except requests.exceptions.HTTPError as e:
        print(f"❌ {e.response.status_code}: {e.response.text[:300]}")
    except Exception as e:
        print(f"❌ {e}")
else:
    print("\n⚠️ Downstream sector ID not found in probe — check Step 2 output and set manually")

In [0]:
# =============================================================================
# REAL EIC API DATA — Oil & Gas sectors (Upstream, Midstream, Downstream)
# =============================================================================

def fetch_all_projects(energy_sector):
    # EIC v1 returns the full sector list; paging.offset and paging.limit are ignored.
    projects = eic_post("/v1/projects/search", {
        "energySector": energy_sector,
        "includeOperators": True,
        "includeContracts": True,
        "includeFid": True,
        "includeAllLocations": True
    })
    names = eic_post("/v1/projectnames/search", {"energySector": energy_sector})
    if not isinstance(projects, list) or not isinstance(names, list):
        raise ValueError(f"EIC sector {energy_sector} did not return full project and name lists")
    project_ids = [project.get("projectId") for project in projects]
    name_ids = [project.get("projectId") for project in names]
    if (None in project_ids or None in name_ids
            or len(project_ids) != len(set(project_ids))
            or len(name_ids) != len(set(name_ids))
            or set(project_ids) != set(name_ids)):
        raise ValueError(f"EIC sector {energy_sector} project IDs do not match its project-name index")
    return projects, len(name_ids)

OG_SECTORS = {"Upstream": 1, "Midstream": 7, "Downstream": 2}
projects_by_sector = {}
for sector_name, sector_id in OG_SECTORS.items():
    sector_projects, name_count = fetch_all_projects(sector_id)
    projects_by_sector[sector_name] = sector_projects
    print(f"{sector_name}: {len(sector_projects)} projects (project-name index: {name_count})")

og_project_ids = {
    project["projectId"]
    for sector_projects in projects_by_sector.values()
    for project in sector_projects
}
print(f"Distinct EIC Oil & Gas project IDs across sectors: {len(og_project_ids)}")

raw_projects = projects_by_sector["Downstream"]
if not raw_projects:
    raise ValueError("EIC returned no Downstream projects; cannot create an empty Spark DataFrame")

# --- Flatten Downstream JSON into the Santiago analysis ---
rows = []
for p in raw_projects:
    locs = p.get("locations", [])
    primary_loc = next((l for l in locs if l.get("main")), locs[0] if locs else {})

    contracts = p.get("contracts", [])
    epc_contractors = [
        c["contractor"]["companyName"]
        for c in contracts
        if c.get("contractor") and "EPC" in c.get("contractType", "")
    ]
    all_contractors = [
        c["contractor"]["companyName"]
        for c in contracts
        if c.get("contractor")
    ]

    op_companies = p.get("operatingCompanies", [])
    operators_list = [o["companyName"] for o in op_companies if o.get("isOperator")]
    partners_list  = [o["companyName"] for o in op_companies if not o.get("isOperator")]

    rows.append({
        "project_id":        int(p["projectId"]),
        "project_name":      str(p.get("projectName") or ""),
        "sector":            str(p.get("energySector") or ""),
        "stage":             str(p.get("projectStage") or ""),
        "status":            str(p.get("projectStatus") or ""),
        "project_type":      str(p.get("projectType") or ""),
        "capex_usd_m":       float(p["value"]) if p.get("value") is not None else None,
        "startup_year":      int(p["startupYear"]) if p.get("startupYear") is not None else None,
        "terrain":           "Offshore" if p.get("isOffshore") else "Onshore",
        "country":           str(primary_loc.get("country") or ""),
        "region":            str(primary_loc.get("region") or ""),
        "locality":          str(primary_loc.get("locality") or ""),
        "operator":          str(p.get("operator") or ""),
        "operators":         ", ".join(operators_list),
        "partners":          ", ".join(partners_list),
        "epc_contractors":   ", ".join(epc_contractors) if epc_contractors else "",
        "all_contractors":   ", ".join(all_contractors) if all_contractors else "",
        "contract_count":    int(len(contracts)),
        "tracking_status":   str(p.get("trackingStatus") or ""),
        "description":       str(p.get("summaryProfile") or "")[:300],
        "source":            "EIC_API_live"
    })

df_eic = spark.createDataFrame(rows)
print("\nSchema:")
df_eic.printSchema()

print("\n--- Downstream summary ---")
df_eic.groupBy("region").count().orderBy("count", ascending=False).show()
df_eic.groupBy("stage").count().orderBy("count", ascending=False).show()

print("\n--- Downstream projects (sorted by CAPEX) ---")
display(df_eic.select(
    "project_id", "project_name", "country", "region", "stage",
    "capex_usd_m", "startup_year", "operator", "epc_contractors", "contract_count"
).orderBy(F.col("capex_usd_m").desc()))

In [0]:
# Distinct Oil & Gas project inventory across all three EIC sectors.
og_projects = {}
for sector_name, sector_projects in projects_by_sector.items():
    for project in sector_projects:
        project_id = project.get("projectId", project.get("projectID"))
        if project_id not in og_projects:
            locations = project.get("locations") or []
            main_location = next((loc for loc in locations if loc.get("main")), locations[0] if locations else {})
            og_projects[project_id] = {
                "project_id": int(project_id),
                "project_name": str(project.get("projectName") or ""),
                "country": str(main_location.get("country") or ""),
                "stage": str(project.get("projectStage") or ""),
                "status": str(project.get("projectStatus") or ""),
                "startup_year": int(project["startupYear"]) if project.get("startupYear") is not None else None,
                "sectors": [sector_name],
            }
        elif sector_name not in og_projects[project_id]["sectors"]:
            og_projects[project_id]["sectors"].append(sector_name)

df_eic_og = spark.createDataFrame([
    {**project, "sectors": ", ".join(project["sectors"])}
    for project in og_projects.values()
])
print(f"Distinct EIC Oil & Gas projects: {df_eic_og.count()}")
display(df_eic_og.orderBy("project_name"))

In [0]:
# Salesforce opportunities from the Synapse-backed Unity Catalog table.
# Opportunity rows are not guaranteed to be unique real-world projects.
sf_og_summary = spark.sql("""
SELECT Etex_Project_Subtype AS subtype,
       COUNT(*) AS opportunity_rows,
       COUNT(DISTINCT BK_Opportunity_Id) AS distinct_opportunities,
       COUNT(DISTINCT Etex_Parent_Project) AS referenced_parent_projects
FROM `synapse-prd`.dwh.f_comm_opportunity
WHERE Etex_Project_Subtype IN ('Oil&Gas', 'Gas')
GROUP BY Etex_Project_Subtype
ORDER BY opportunity_rows DESC
""")
display(sf_og_summary)

sf_oil_gas = spark.sql("""
SELECT BK_Opportunity_Id AS opportunity_id,
       Name AS opportunity_name,
       Etex_Parent_Project AS parent_project_id,
       Etex_Country AS country,
       Etex_Region AS region,
       Account_Id AS account_id,
       Stage_Name AS stage,
       Etex_Project_Type AS project_type,
       Etex_Project_Subtype AS project_subtype
FROM `synapse-prd`.dwh.f_comm_opportunity
WHERE Etex_Project_Subtype IN ('Oil&Gas', 'Gas')
""")
display(sf_oil_gas.orderBy('opportunity_name'))

In [0]:
# EIC -> Salesforce opportunity review. Run after the EIC inventory and CRM cells.
import csv
import json
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from databricks.sdk import WorkspaceClient
from pyspark.sql.types import ArrayType, BooleanType, DoubleType, IntegerType, StringType, StructField, StructType
from og_matching import databricks_llm_decider, match_projects

project_records = [
    {**project, "sectors": ", ".join(project["sectors"])}
    for project in og_projects.values()
]
opportunity_records = [row.asDict(recursive=True) for row in sf_oil_gas.collect()]
input_by_subtype = dict(Counter(row["project_subtype"] for row in opportunity_records))
assert set(input_by_subtype) == {"Oil&Gas", "Gas"}, input_by_subtype
match_rows, match_summary = match_projects(
    project_records,
    opportunity_records,
    llm_decider=databricks_llm_decider(WorkspaceClient()),
    max_llm_calls=50,
)
assert len(match_rows) == len(og_projects)

match_schema = StructType([
    StructField("project_id", IntegerType(), False),
    StructField("project_name", StringType(), True),
    StructField("country", StringType(), True),
    StructField("sectors", StringType(), True),
    StructField("opportunity_id", StringType(), True),
    StructField("opportunity_name", StringType(), True),
    StructField("opportunity_subtype", StringType(), True),
    StructField("candidate_ids", ArrayType(StringType()), True),
    StructField("candidate_names", ArrayType(StringType()), True),
    StructField("candidate_countries", ArrayType(StringType()), True),
    StructField("candidate_subtypes", ArrayType(StringType()), True),
    StructField("candidate_scores", ArrayType(DoubleType()), True),
    StructField("best_candidate_score", DoubleType(), True),
    StructField("match_status", StringType(), False),
    StructField("review_required", BooleanType(), False),
    StructField("decision_source", StringType(), False),
    StructField("reason", StringType(), False),
])
df_matches = spark.createDataFrame(match_rows, schema=match_schema)

output_dir = Path("/Workspace/Users/mustafaarda.bicakci@etexgroup.com/Initiatives/2492_OG_Market_Intelligence/match_outputs")
output_dir.mkdir(parents=True, exist_ok=True)
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
review_file = output_dir / f"eic_sf_review_{run_id}.csv"
summary_file = output_dir / f"eic_sf_summary_{run_id}.json"

def csv_value(value):
    text = json.dumps(value, ensure_ascii=False) if isinstance(value, list) else str(value if value is not None else "")
    return "'" + text if text.lstrip().startswith(("=", "+", "-", "@")) else text

with review_file.open("w", newline="", encoding="utf-8") as output:
    writer = csv.DictWriter(output, fieldnames=list(match_rows[0]))
    writer.writeheader()
    for row in match_rows:
        writer.writerow({key: csv_value(value) for key, value in row.items()})
summary_file.write_text(json.dumps({
    **match_summary,
    "run_utc": run_id,
    "model_endpoint": "databricks-gpt-5-4-mini",
    "salesforce_subtypes": input_by_subtype,
    "input_opportunities": len(opportunity_records),
    "policy": "LLM proposals require human review; New is not proof of absence",
}, indent=2), encoding="utf-8")
print(f"Review file: {review_file}\nRun summary: {summary_file}\n{match_summary}")
display(df_matches.groupBy("match_status", "review_required").count())
display(df_matches.filter("review_required").select("project_id", "project_name", "country", "opportunity_id", "opportunity_subtype", "candidate_names", "candidate_subtypes", "candidate_scores", "match_status", "reason").limit(100))

In [0]:
# Publish the same verified EIC/CRM snapshot as governed, queryable UC data.
from og_data_product import publish_snapshot

publish_snapshot(spark, run_id, df_eic_og, sf_oil_gas, df_matches,
                 match_summary, input_by_subtype)
print(f"Published completed OG snapshot {run_id} to automation_s.og")

## Current verification status (28 September 2026)

- The notebook loaded `EIC-API` from the Key Vault-backed Databricks scope `WE-SPOKE-MDP-AUT-T-KV` and completed a Databricks end-to-end run. No API key or saved output is stored in this notebook.
- Live EIC: Upstream (1) 1,747; Midstream (7) 1,177; Downstream (2) 1,424; 4,348 distinct project IDs. For each sector the project and project-name ID sets agreed. EIC v1 ignored paging inputs, so the notebook fetches each sector once. The older saved 34-Downstream result is obsolete.
- With `includeAllLocations`, none of these records lacks a name, country, CAPEX or stage. Startup year is absent on 99 Upstream, 7 Midstream and 6 Downstream projects; the current Downstream year range is 2026–2040. Business validation of year accuracy is still required.
- Salesforce opportunities are available through `synapse-prd.dwh.f_comm_opportunity`. `Oil&Gas` returns 1,141 distinct opportunity IDs; `Gas` is a separate subtype with 68. Opportunity rows are not project counts; `Oil&Gas` opportunities refer to 34 distinct non-null parent-project IDs.
- The LATAM/Santiago view remains Downstream-only. Old June 2026 planning sections are historical. Production coverage, acceptable-use terms, startup-year accuracy and the 1% revenue assumption need business verification.
- A verified EIC HTTPS request fails certificate validation in Databricks serverless. The notebook's `verify=False` workaround is for the POC only; install the corporate CA bundle and restore TLS verification before production.

In [0]:
# =============================================================================
# SANTIAGO'S PIPELINE — LATAM Downstream Projects
# Filters: LATAM countries | FEED or EPC stage | year filter pending validation
# Output: POC Promat revenue potential per project (unapproved 1% assumption)
# =============================================================================

# --- LATAM country list (EIC uses 'South America' region + Mexico) ---
LATAM_COUNTRIES = [
    "Brazil", "Argentina", "Colombia", "Chile", "Mexico", "Peru",
    "Bolivia", "Ecuador", "Venezuela", "Uruguay", "Paraguay",
    "Panama", "Costa Rica", "Guatemala", "Honduras", "El Salvador",
    "Nicaragua", "Dominican Republic", "Cuba", "Trinidad and Tobago"
]
LATAM_REGIONS = ["South America", "Central America", "Caribbean"]
# Note: EIC puts Mexico under 'North America' region — filter by country name instead

# --- Promat revenue estimate: ~1% of CAPEX (passive fire protection)
# TODO: Confirm exact % with Santiago (he has his own formula)
PRO_REVENUE_PCT = 0.01

# --- Apply filters ---
df_santiago = (
    df_eic
    .filter(
        (F.col("country").isin(LATAM_COUNTRIES)) |
        (F.col("region").isin(LATAM_REGIONS))
    )
    .filter(F.col("stage").isin(["FEED", "EPC"]))
    # .filter(F.col("startup_year") >= 2027)  # Enable only after year accuracy is approved
    .withColumn("promat_revenue_est_m", F.round(F.col("capex_usd_m") * PRO_REVENUE_PCT, 2))
    .orderBy(F.col("capex_usd_m").desc())
)

print(f"LATAM Downstream FEED/EPC projects: {df_santiago.count()}")
print("Startup-year filter disabled pending business validation of EIC dates")

# --- Summary ---
print("\nBy country:")
df_santiago.groupBy("country").agg(
    F.count("*").alias("projects"),
    F.sum("capex_usd_m").alias("total_capex_m"),
    F.sum("promat_revenue_est_m").alias("promat_potential_m")
).orderBy(F.col("total_capex_m").desc()).show(truncate=False)

print(f"Total CAPEX pipeline: ${df_santiago.agg(F.sum('capex_usd_m')).collect()[0][0]:,.0f}M")
print(f"Promat revenue potential (@{PRO_REVENUE_PCT*100:.0f}% CAPEX): ${df_santiago.agg(F.sum('promat_revenue_est_m')).collect()[0][0]:,.0f}M")

# --- Full table ---
display(df_santiago.select(
    "project_id", "project_name", "country", "region", "stage",
    "capex_usd_m", "promat_revenue_est_m", "startup_year",
    "operator", "epc_contractors", "contract_count"
))

## Current Workflow

### How it works today:
```
1. Log into EIC DataStream
2. Filter by region (e.g. South America → Brazil)
3. Select sector (e.g. Downstream = refineries)
4. Filter by startup year (e.g. 2027)
5. Filter by stage (FEED, EPC)
6. View CAPEX per project
7. Click project → identify stakeholders (EPCs, operators)
8. Calculate Promat revenue potential from CAPEX
9. Manually enter project into Salesforce
10. Assign to commercial team member by region
```

### What he wants automated:
| Current (Manual) | Future (Automated) |
| --- | --- |
| Log into EIC, filter, browse | Auto-ingest all projects via API |
| Check BN Americas separately | Combine multiple sources |
| Google search for updates | AI agent finds changes |
| Manually identify stakeholders | AI extracts EPC/operator names |
| Estimate revenue from CAPEX | Auto-calculate Promat potential |
| Manually type into Salesforce | Auto-push to Salesforce |
| One person researches one project | AI covers thousands simultaneously |
| React to known projects | Proactively detect NEW projects early |



### Revenue Calculation Logic:
> "If this is a $500 million project, I have a calculation to know how much of this will represent to passive fire protection based on past experience."

This implies a **CAPEX → Promat Revenue %** formula exists internally that we should integrate.

## Dependencies & Blockers — Updated June 26, 2026

### ✅ Resolved
| Dependency | Status | Notes |
| --- | --- | --- |
| EIC API Test Key | ✅ DONE | Test key obtained June 25, 2026 ("Test v.1. Automation - Arda - Etex") |
| API connectivity | ✅ CONFIRMED | `https://memberapi.the-eic.com`, `Authorization: key <apiKey>` |
| JSON response structure | ✅ CONFIRMED | All fields validated: projectId, value, locations[], contracts[], operatingCompanies[] |
| Pagination model | ✅ CONFIRMED | POST body: `{paging: {limit: N, offset: N}}` |
| Contractor nesting | ✅ CONFIRMED | `contracts[].contractor.companyName` + bidStatus |
| Downstream completeness | ✅ CONFIRMED | 34 projects, full structure with operators + EPC contractors |
| Sector ID mapping | ✅ CONFIRMED | Downstream = sector_id 2 |
| CAPEX field | ✅ CONFIRMED | Field: `value` (USD millions) |

### 🔴 Critical Blockers Remaining
| Dependency | Status | Owner | Action Required |
| --- | --- | --- | --- |
| Production EIC Key | ⏳ In progress | Santiago + EIC | Needed for real project names & correct startup years |
| EIC Contract / Terms | ⏳ In progress | Santiago / Bernard | Sign updated membership contract |
| Salesforce Data Access | ❌ Not started | Santiago | Read access to Opportunities (see options A/B/C in doc) |

### 🟡 To Be Confirmed
| Dependency | Status | Owner | Notes |
| --- | --- | --- | --- |
| Promat CAPEX→Revenue formula | ❓ Unknown | Santiago | POC uses 1% placeholder — confirm real % |
| MVP sector scope | ❓ Unknown | Santiago | Downstream confirmed; others TBC |
| MVP region scope | ❓ Unknown | Santiago | Global or priority regions? |
| API rate limits | ❓ Unknown | EIC / Zoe Hadwani | Not yet tested |

---

## Recommended Next Steps (Updated)

1. ✅ EIC API test key — DONE
2. ✅ API connectivity & response structure validated — DONE
3. ✅ Downstream data ingestion POC running — DONE
4. 🔲 Build Bronze Delta table (scheduled ingestion, all sectors)
5. 🔲 Build Silver transformation layer
6. 🔲 Confirm sectors / stages / regions with Santiago → apply filters
7. 🔲 Salesforce data access (path A/B/C)
8. 🔲 Fuzzy matching algorithm (EIC ↔ Salesforce)
9. 🔲 Gold unified view + dashboard
10. 🔲 Production EIC key → enable real project names + startup years

## Target Delta Table Schema

### Bronze Layer (Raw API Response)
```sql
CREATE TABLE IF NOT EXISTS bronze.eic_raw_projects (
  raw_json STRING,                    -- Full JSON response
  project_id STRING,                  -- EIC project ID (extracted for partitioning)
  ingestion_timestamp TIMESTAMP,      -- When we pulled it
  api_endpoint STRING,                -- Which endpoint was called
  api_version STRING,                 -- API version if available
  batch_id STRING                     -- Ingestion batch identifier
)
USING DELTA
PARTITIONED BY (ingestion_timestamp)
```

### Silver Layer (Standardized Project Model)
```sql
CREATE TABLE IF NOT EXISTS silver.og_projects (
  project_id STRING,                  -- EIC unique ID
  project_name STRING,                -- Normalized project name
  country STRING,                     -- Country
  region STRING,                      -- Geographic region
  sector STRING,                      -- Downstream/Upstream/LNG/Hydrogen/etc.
  stage STRING,                       -- Conceptual/FEED/EPC/Construction
  project_type STRING,                -- Refinery/LNG Terminal/etc.
  capex_usd_millions DOUBLE,          -- CAPEX in USD millions
  startup_year INT,                   -- Expected startup year
  operator_name STRING,               -- Operating company
  contractor_names ARRAY<STRING>,     -- EPC/construction companies
  terrain STRING,                     -- Onshore/Offshore
  water_depth_m DOUBLE,               -- Offshore depth
  fid_date DATE,                      -- Final Investment Decision date
  capacity DOUBLE,                    -- Production capacity
  capacity_unit STRING,               -- Unit (MW, bpd, mtpa, etc.)
  description STRING,                 -- Project description
  last_updated DATE,                  -- Source last update
  source STRING,                      -- 'EIC' / 'BN_Americas' / etc.
  promat_potential_eur DOUBLE,        -- Calculated Promat revenue potential
  -- Metadata
  ingestion_date DATE,
  silver_processed_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (region, sector)
```

### Gold Layer (Unified Intelligence + Salesforce Match)
```sql
CREATE TABLE IF NOT EXISTS gold.og_project_intelligence (
  project_id STRING,
  project_name STRING,
  country STRING,
  region STRING,
  sector STRING,
  stage STRING,
  capex_usd_millions DOUBLE,
  startup_year INT,
  operator_name STRING,
  contractor_names ARRAY<STRING>,
  promat_potential_eur DOUBLE,
  -- Salesforce matching
  sf_match_status STRING,             -- 'Matched' / 'New' / 'Partial' / 'No Match'
  sf_opportunity_id STRING,           -- Salesforce Opportunity ID if matched
  sf_opportunity_name STRING,         -- Salesforce name
  sf_owner STRING,                    -- Salesforce opportunity owner
  sf_stage STRING,                    -- Salesforce stage
  match_confidence DOUBLE,            -- 0.0 - 1.0 fuzzy match score
  match_method STRING,                -- 'exact' / 'fuzzy_name' / 'stakeholder' / 'country+sector'
  -- Action tracking
  assigned_to STRING,                 -- Commercial team member
  priority_score DOUBLE,              -- Calculated priority
  last_action_date DATE,
  notes STRING,
  -- Metadata
  source STRING,
  last_updated DATE,
  gold_processed_at TIMESTAMP
)
USING DELTA
PARTITIONED BY (region)
```

# 📋 Stakeholder Document Update — June 26, 2026
*Sections below replace / supplement the original scope document sent in June 2026.*

---

## UPDATE: Section 3.1 — API Unknowns Resolved

The following items from the original "still need to be validated" list are now confirmed:

| Item | Was | Now |
| --- | --- | --- |
| JSON response structure | Unknown | ✅ Confirmed (see fields below) |
| Pagination model | Unknown | ✅ POST body `{paging: {limit, offset}}` |
| Auth format | Unknown | ✅ `Authorization: key <apiKey>` |
| Contractor nesting | Unknown | ✅ `contracts[].contractor.companyName` + bidStatus |
| Downstream completeness | Unknown | ✅ Validated: full structure with operators, EPC contractors, CAPEX |
| CAPEX field | Unknown | ✅ Field `value` in USD millions, present for all projects |
| Rate limits | Unknown | Not yet tested — low risk, assess during Sprint 1 |

**Confirmed response fields per project:**
`projectId` · `projectName` · `energySector` · `value` (CAPEX USD M) · `startupYear` · `projectStage` · `projectStatus` · `projectType` · `isOffshore` · `operator` · `summaryProfile` · `locations[]` (country, region, locality) · `operatingCompanies[]` (name, isOperator, equityShare) · `contracts[]` (contractType, contractStage, awardDate, contractor.companyName, bidStatus)

**Test API note:** Project names are anonymised in the test environment. Startup years are incorrect in test data. Both will be accurate with the production key.

---

## UPDATE: Section 6 — Risks & Dependencies

| Risk / Dependency | Was | Now |
| --- | --- | --- |
| EIC test key not available | 🔴 Critical blocker | ✅ Resolved — key obtained June 25, 2026 |
| API terms not approved | 🔴 Critical | ⏳ In progress with Zoe Hadwani / Nigel O’Sullivan (EIC) |
| EPC/contractor data structure unclear | 🟡 Risk | ✅ Resolved — structure fully validated |
| CAPEX completeness lower than expected | 🟡 Risk | ✅ Resolved — CAPEX present for all downstream projects |
| Salesforce access delayed | 🟡 Risk | Still open — Excel export (Option C) recommended to unblock |

---

## UPDATE: Section 8 — Immediate Next Steps

| # | Action | Owner | Status |
| --- | --- | --- | --- |
| 1 | EIC API test key | Santiago | ✅ DONE — June 25, 2026 |
| 2 | Confirm API terms / acceptable use | Santiago / IT | ⏳ In progress |
| 3 | Share Salesforce project data | Santiago | 🔲 Not started |
| 4 | Sample Salesforce export if API delayed | Santiago / SF owner | 🔲 Not started |
| 5 | Confirm MVP sectors and stages | Santiago | 🔲 Needed |
| 6 | Confirm CAPEX-to-Promat-potential % | Santiago | 🔲 Needed (POC uses 1% placeholder) |
| 7 | Confirm region / team routing logic | Santiago | 🔲 Needed |
| 8 | Production EIC key (replaces test key) | Santiago + EIC | 🔲 Needed for real project names |

---

## NEW: Section 9 — Development Timeline & Effort Estimation

### Assumptions
- Data engineer: 1 person (Arda)
- Salesforce access available within 1 week of next steps meeting
- Sector/stage/region confirmed by Santiago within 1 week
- CAPEX formula confirmed before Gold layer

### Sprint Plan

| Sprint | Scope | Effort | Calendar | Dependency |
| --- | --- | --- | --- | --- |
| **Sprint 0** — EIC POC | API connection, auth, sector discovery, downstream ingestion (34 test projects), LATAM filter, Promat revenue calc | ✅ **Done** | June 25–26, 2026 | Test key |
| **Sprint 1** — EIC Production Ingestion | Bronze Delta table (all sectors, scheduled weekly), Silver normalization, sector/stage/region filters, CAPEX formula integration, incremental refresh (`updatedSince`) | **5 days** | Week 1 | Production key + sector/stage confirmation |
| **Sprint 2** — Salesforce Integration | Salesforce staging layer, field normalization, fuzzy matching algorithm (project name + country + operator), confidence scoring | **6 days** | Week 2–3 | Salesforce data access |
| **Sprint 3** — Gold Layer + Output | Gold unified view (EIC + Salesforce), match status (Matched / New / Partial / Uncertain), Promat revenue potential column, region→team assignment | **4 days** | Week 3–4 | Sprint 1 + 2 |
| **Sprint 4** — Dashboard & Validation | Databricks dashboard or Excel export, business validation with Santiago, false positive review, acceptance testing | **4 days** | Week 4–5 | Sprint 3 |
| **Buffer** | Bug fixes, business feedback, iteration | **2 days** | Week 5 | — |

### Total Effort
| Component | Days |
| --- | --- |
| EIC ingestion (Bronze + Silver + scheduling) | 5 |
| Salesforce staging + matching | 6 |
| Gold layer + unified view | 4 |
| Dashboard + validation | 4 |
| Buffer | 2 |
| **Total** | **21 days (~5 weeks)** |

### Timeline Summary
```
Week 0 (done):  EIC API POC ✓
Week 1:         Bronze + Silver ingestion  [needs: production key, sector confirmation]
Week 2-3:       Salesforce staging + matching  [needs: Salesforce data]
Week 3-4:       Gold layer + unified view
Week 4-5:       Dashboard + business validation
└─ MVP ready:   ~5 weeks from Salesforce access date
```

### Critical Path
The single biggest risk to the timeline is **Salesforce data access**. If Santiago provides an Excel export (Option C) within 1 week, the MVP can be delivered in 5 weeks from today. If Salesforce access is delayed beyond 2 weeks, the MVP timeline shifts accordingly.

---

## What the MVP Will Deliver

| Deliverable | Included in MVP |
| --- | --- |
| All EIC downstream (+ confirmed sectors) projects, refreshed weekly | ✅ |
| Standardized project model: name, country, sector, stage, CAPEX, startup year, operator, EPC contractors | ✅ |
| Promat revenue potential (once formula confirmed) | ✅ |
| EIC vs Salesforce match status per project | ✅ |
| New opportunities not yet in Salesforce | ✅ |
| Reviewed output usable by Santiago for follow-up | ✅ |
| Salesforce writeback (auto-create opportunities) | ❌ Future phase |
| AI enrichment / public source enrichment | ❌ Future phase |
| BN Americas integration | ❌ Future phase |

# 📋 Stakeholder Document Update — June 26, 2026
*Sections below replace / supplement the original scope document sent in June 2026.*

---

## UPDATE: Section 3.1 — API Unknowns Resolved

The following items from the original "still need to be validated" list are now confirmed:

| Item | Was | Now |
| --- | --- | --- |
| JSON response structure | Unknown | ✅ Confirmed (see fields below) |
| Pagination model | Unknown | ✅ POST body `{paging: {limit, offset}}` |
| Auth format | Unknown | ✅ `Authorization: key <apiKey>` |
| Contractor nesting | Unknown | ✅ `contracts[].contractor.companyName` + bidStatus |
| Downstream completeness | Unknown | ✅ Validated: full structure with operators, EPC contractors, CAPEX |
| CAPEX field | Unknown | ✅ Field `value` in USD millions, present for all projects |
| Rate limits | Unknown | Not yet tested — low risk, assess in Sprint 1 |

**Confirmed response fields per project:**
`projectId` · `projectName` · `energySector` · `value` (CAPEX USD M) · `startupYear` · `projectStage` · `projectStatus` · `projectType` · `isOffshore` · `operator` · `summaryProfile` · `locations[]` (country, region, locality) · `operatingCompanies[]` (name, isOperator, equityShare) · `contracts[]` (contractType, contractStage, awardDate, contractor.companyName, bidStatus)

> **Test API note:** Project names are anonymised in the test environment and startup years are incorrect. Both will be accurate with the production key.

---

## UPDATE: Section 6 — Risks & Dependencies

| Risk / Dependency | Was | Now |
| --- | --- | --- |
| EIC test key not available | 🔴 Critical blocker | ✅ Resolved — obtained June 25, 2026 |
| API terms not approved | 🔴 Critical | ⏳ In progress with Zoe Hadwani / Nigel O’Sullivan (EIC) |
| EPC/contractor data structure unclear | 🟡 Risk | ✅ Resolved — fully validated |
| CAPEX completeness lower than expected | 🟡 Risk | ✅ Resolved — CAPEX present for all downstream projects |
| Salesforce access delayed | 🟡 Risk | Still open — Excel export (Option C) recommended to unblock |

---

## UPDATE: Section 8 — Immediate Next Steps

| # | Action | Owner | Status |
| --- | --- | --- | --- |
| 1 | EIC API test key | Santiago | ✅ DONE |
| 2 | Confirm API terms / acceptable use | Santiago / IT | ⏳ In progress |
| 3 | Share Salesforce project data | Santiago | 🔲 Pending |
| 4 | Sample Salesforce export if API delayed | Santiago / SF owner | 🔲 Pending |
| 5 | Confirm MVP sectors and stages | Santiago | 🔲 Needed |
| 6 | Confirm CAPEX-to-Promat % formula | Santiago | 🔲 Needed (POC uses 1% placeholder) |
| 7 | Confirm region / team routing logic | Santiago | 🔲 Needed |
| 8 | Production EIC key | Santiago + EIC | 🔲 Needed for real project names + startup years |

---

## NEW: Section 9 — Development Timeline & Effort

### Assumptions
- 1 data engineer (Arda Biçakçi)
- Salesforce data available within 1 week of next steps meeting
- Sector/stage/region scope confirmed by Santiago within 1 week
- CAPEX formula confirmed before Gold layer build

### Sprint Plan

| Sprint | Scope | Effort | Calendar | Key Dependency |
| --- | --- | --- | --- | --- |
| **Sprint 0 — EIC POC** | API connection, auth, sector discovery, full downstream ingestion, LATAM filter, Promat revenue POC | ✅ **Done** | June 25–26 | Test key |
| **Sprint 1 — EIC Data Pipeline (Bronze + Silver)** | Bronze Delta table (all confirmed sectors), Silver normalization, stage/region filters, CAPEX formula, incremental refresh via `updatedSince`, scheduled Databricks job | **8h** | Week 1 | Production key + scope confirmation |
| **Sprint 2 — Salesforce Integration** | Salesforce staging layer, field normalization, fuzzy matching algorithm (`rapidfuzz`), confidence scoring | **16h** | Week 2 | Salesforce data access |
| **Sprint 3 — Gold Layer + Output** | Unified view (EIC + Salesforce), match status (Matched / New / Partial / Uncertain), Promat revenue column, region→team routing | **8h** | Week 2–3 | Sprint 1 + 2 done |
| **Sprint 4a — Developer Validation** | Developer acceptance testing, data quality checks, edge case review | **8h** | Week 3 | Sprint 3 done |
| **Sprint 4b — Dashboard (optional)** | Output dashboard or Excel export for business review | **8h** | Week 3 | Sprint 4a done |
| **User Validation** | Business review with Santiago — confirm output is actionable, false positive check | TBD | Week 3+ | Sprint 4a done |
| **Production** | Move to production environment, secrets to Key Vault, monitoring setup | **8h** | Week 4 | User validation done |

### Effort Summary

| Component | Days |
| --- | --- |
| EIC Data Pipeline (Bronze + Silver + job scheduling) | 8h |
| Salesforce staging + fuzzy matching | 16h |
| Gold layer + unified view | 8h |
| Developer validation | 8h |
| Dashboard (optional) | 8h |
| Production deployment | 8h |
| **Total dev effort** | **56 hours** |
| User validation | TBD (Santiago) |
| CI/CD setup (Azure DevOps + DABs) | Future phase |

### Timeline
```
Week 0  (DONE)    EIC API POC ✓
Week 1            Bronze + Silver ingestion  ← needs production key + scope
Week 2            Salesforce staging + fuzzy matching  ← needs Salesforce data
Week 2–3          Gold layer + unified view
Week 3            Developer validation + Dashboard (optional)
                  └─ MVP delivery
```

**Critical path:** Salesforce data access. An Excel export (Option C) within 1 week of the next meeting delivers MVP in 2 weeks.

---

## MVP Scope Confirmation

| Deliverable | MVP |
| --- | --- |
| EIC projects (confirmed sectors), weekly refresh | ✅ |
| Standardized project model: name, country, sector, stage, CAPEX, startup year, operator, EPC contractors | ✅ |
| Promat revenue potential (once formula confirmed) | ✅ |
| EIC vs Salesforce match status per project | ✅ |
| New opportunities not yet in Salesforce | ✅ |
| Santiago-reviewed actionable output | ✅ |
| Salesforce writeback (auto-create opportunities) | ❌ Future phase |
| AI enrichment / public source enrichment | ❌ Future phase |
| BN Americas integration | ❌ Future phase |